# Phase 1 matched-order XNLI run (Kaggle): EN -> HI -> DE

This notebook runs the matched English -> Hindi -> German arm: `facebook/xnli`, no replay, 1,000 training and 500 test examples per language, fixed test-selection seed 42, and training seeds 42, 43, and 44. It uses the same model, NLI prompt, and LoRA settings as the Hindi -> German -> English comparison arm.

## Kaggle setup

1. Add `phase1_xnli_multiseed_colab_package.zip` to a Kaggle Dataset and attach it as notebook input. The notebook accepts the ZIP or its extracted folder.
2. Enable Internet and a GPU (T4/P100 or better; availability depends on Kaggle quota).
3. Accept access to `meta-llama/Llama-3.2-1B` and add a Kaggle secret named `HF_TOKEN`.
4. Run all cells. Three seeds run sequentially and may take several hours. Results and checkpoints are saved under `/kaggle/working/phase1_en_hi_de/`; the final cell creates a downloadable ZIP.

W&B is disabled; no W&B secret is needed. The archive contains evaluation records, summaries, and LoRA checkpoints, but not raw XNLI examples or credentials. Keep the output private. This is a new matched run, separate from the older 500/200 pilot; training is not started until you run the notebook.


In [ ]:
%pip -q install "datasets>=3.0" "transformers>=4.46" "peft>=0.13" accelerate bitsandbytes


In [ ]:
import os
import torch
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU found. Enable a GPU in Kaggle Notebook options, then rerun this cell.")

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Add a Kaggle secret named HF_TOKEN before running this notebook.")
os.environ["HF_TOKEN"] = hf_token
login(token=hf_token, add_to_git_credential=False)
del hf_token

print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
print("Hugging Face authentication completed; credentials were not printed.")


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import sys

INPUT_ROOT = Path("/kaggle/input")
SOURCE_ARCHIVE_NAME = "phase1_xnli_multiseed_colab_package.zip"
source_archives = sorted(path for path in INPUT_ROOT.rglob(SOURCE_ARCHIVE_NAME) if path.is_file())
if len(source_archives) > 1:
    raise FileNotFoundError(f"Found duplicate source archives named {SOURCE_ARCHIVE_NAME}: {source_archives}")

if source_archives:
    SOURCE_PACKAGE_PATH = source_archives[0]
    PACKAGE_ROOT = Path("/kaggle/working/phase1_source_package")
    if not PACKAGE_ROOT.exists():
        PACKAGE_ROOT.mkdir(parents=True)
        with ZipFile(SOURCE_PACKAGE_PATH) as archive:
            root = PACKAGE_ROOT.resolve()
            for member in archive.infolist():
                destination = (PACKAGE_ROOT / member.filename).resolve()
                if destination != root and root not in destination.parents:
                    raise ValueError("The source ZIP contains an unsafe path.")
            archive.extractall(PACKAGE_ROOT)
else:
    experiment_files = sorted(
        path for path in INPUT_ROOT.rglob("experiment.py")
        if path.parent.name == "continual_learning_lora"
    )
    package_roots = sorted({path.parents[2] for path in experiment_files})
    if len(package_roots) != 1:
        raise FileNotFoundError(
            f"Expected one {SOURCE_ARCHIVE_NAME} or one extracted Phase 1 source package under {INPUT_ROOT}; "
            f"found {package_roots}. Attach the source-package Kaggle Dataset."
        )
    SOURCE_PACKAGE_PATH = package_roots[0]
    PACKAGE_ROOT = SOURCE_PACKAGE_PATH

required_source = PACKAGE_ROOT / "shared" / "continual_learning_lora" / "experiment.py"
if not required_source.is_file():
    raise FileNotFoundError(f"The Phase 1 experiment package was not found in {PACKAGE_ROOT}.")
sys.path.insert(0, str(PACKAGE_ROOT))
print("Phase 1 source package:", SOURCE_PACKAGE_PATH)


In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path
from datasets import load_dataset

SEEDS = [42, 43, 44]
TEST_SPLIT_SEED = 42
LANGUAGES = ["en", "hi", "de"]
TRAIN_SAMPLES_PER_LANGUAGE = 1000
TEST_SAMPLES_PER_LANGUAGE = 500
DATASET_ID = "facebook/xnli"
MODEL_ID = "meta-llama/Llama-3.2-1B"
LABEL_NAMES = {0: "entailment", 1: "neutral", 2: "contradiction"}

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
GROUP_ID = (
    f"xnli-en-hi-de-multiseed-train{TRAIN_SAMPLES_PER_LANGUAGE}"
    f"-test{TEST_SAMPLES_PER_LANGUAGE}-{RUN_TIMESTAMP}"
)
GROUP_OUTPUT = Path("/kaggle/working/phase1_en_hi_de") / GROUP_ID
if GROUP_OUTPUT.exists():
    raise FileExistsError(f"Refusing to overwrite an existing run group: {GROUP_OUTPUT}")
(GROUP_OUTPUT / "results").mkdir(parents=True)
(GROUP_OUTPUT / "checkpoints").mkdir()

RUN_CONFIG = {
    "experiment_group_id": GROUP_ID,
    "dataset": DATASET_ID,
    "task_order": LANGUAGES,
    "seeds": SEEDS,
    "train_examples_per_language": TRAIN_SAMPLES_PER_LANGUAGE,
    "test_examples_per_language": TEST_SAMPLES_PER_LANGUAGE,
    "fixed_test_split_seed": TEST_SPLIT_SEED,
    "model": MODEL_ID,
    "method": "sequential_finetuning_no_replay",
    "prompt_protocol": "phase1_b0_nli_prompt_candidate_label_log_likelihood",
    "training": {
        "task_mode": "causal_lm",
        "max_length": 512,
        "learning_rate": 2e-4,
        "train_batch_size": 1,
        "eval_batch_size": 1,
        "gradient_accumulation_steps": 8,
        "num_train_epochs": 1,
        "lora_rank": 8,
        "lora_alpha": 16,
        "lora_dropout": 0.05,
        "load_in_4bit": True,
        "gradient_checkpointing": True,
    },
}
(GROUP_OUTPUT / "run_config.json").write_text(
    json.dumps(RUN_CONFIG, indent=2, allow_nan=False) + "\n", encoding="utf-8"
)


def format_xnli_row(example):
    label_id = int(example["label"])
    if label_id not in LABEL_NAMES:
        raise ValueError(f"Unexpected XNLI label id: {label_id}")
    prompt = (
        "Task: Natural language inference\n"
        f"Premise: {example['premise']}\n"
        f"Hypothesis: {example['hypothesis']}\n"
        "Choose one label: entailment, neutral, contradiction\n"
        "Answer:"
    )
    return {"prompt": prompt, "target": " " + LABEL_NAMES[label_id]}

train_sets_by_seed = {seed: {} for seed in SEEDS}
test_sets = {}
for language in LANGUAGES:
    train_raw = load_dataset(DATASET_ID, language, split="train")
    test_raw = load_dataset(DATASET_ID, language, split="test")
    if len(train_raw) < TRAIN_SAMPLES_PER_LANGUAGE:
        raise ValueError(
            f"Not enough training examples for {language}: "
            f"found {len(train_raw)}, need {TRAIN_SAMPLES_PER_LANGUAGE}."
        )
    if len(test_raw) < TEST_SAMPLES_PER_LANGUAGE:
        raise ValueError(
            f"Not enough test examples for {language}: "
            f"found {len(test_raw)}, need {TEST_SAMPLES_PER_LANGUAGE}."
        )
    test_sets[language] = test_raw.shuffle(seed=TEST_SPLIT_SEED).select(
        range(TEST_SAMPLES_PER_LANGUAGE)
    )
    for seed in SEEDS:
        train_subset = train_raw.shuffle(seed=seed).select(
            range(TRAIN_SAMPLES_PER_LANGUAGE)
        )
        train_sets_by_seed[seed][language] = train_subset.map(
            format_xnli_row,
            remove_columns=train_subset.column_names,
            desc=f"Formatting {language} training examples for seed {seed}",
        )
        if len(train_sets_by_seed[seed][language]) != TRAIN_SAMPLES_PER_LANGUAGE:
            raise ValueError(f"Incorrect training sample count for {language}, seed {seed}.")
    print(
        f"{language}: train={TRAIN_SAMPLES_PER_LANGUAGE} per seed, "
        f"fixed test={len(test_sets[language])}"
    )

if any(len(test_sets[language]) != TEST_SAMPLES_PER_LANGUAGE for language in LANGUAGES):
    raise ValueError("At least one fixed test subset has an incorrect sample count.")
if any(list(train_sets_by_seed[seed]) != LANGUAGES for seed in SEEDS):
    raise ValueError("The prepared training datasets do not match the requested stream order.")
print("Dataset sizes, selection seeds, and prompt formatting checks passed.")


In [ ]:
import gc
import torch
from transformers import set_seed
from shared.continual_learning_lora import LlamaLoRATrainer, run_sequential_experiment

seed_results = []
for seed in SEEDS:
    set_seed(seed)
    experiment_id = f"{GROUP_ID}-seed{seed}"
    checkpoint_dir = GROUP_OUTPUT / "checkpoints" / f"seed-{seed}"
    evaluation_dir = GROUP_OUTPUT / "results" / f"seed-{seed}"
    if checkpoint_dir.exists() or evaluation_dir.exists():
        raise FileExistsError(f"Refusing to reuse output paths for seed {seed}.")
    checkpoint_dir.mkdir()
    evaluation_dir.mkdir()

    trainer = LlamaLoRATrainer(
        model_name=MODEL_ID,
        output_dir=checkpoint_dir,
        task_mode="causal_lm",
        max_length=512,
        learning_rate=2e-4,
        train_batch_size=1,
        eval_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=1,
        seed=seed,
        lora_rank=8,
        lora_alpha=16,
        lora_dropout=0.05,
        load_in_4bit=True,
        gradient_checkpointing=True,
        token=True,
    )
    trainer.load_model_and_tokenizer()
    print(f"Starting seed {seed}; trainable parameter summary:")
    trainer.model.print_trainable_parameters()
    result = run_sequential_experiment(
        trainer,
        task_order=LANGUAGES,
        train_datasets=train_sets_by_seed[seed],
        test_datasets=test_sets,
        experiment_id=experiment_id,
        output_dir=evaluation_dir,
        dataset_name=DATASET_ID,
        task_name="xnli",
        run_metadata={
            "phase": 1,
            "method": "sequential_finetuning_no_replay",
            "seed": seed,
            "seed_group": GROUP_ID,
            "test_split_seed": TEST_SPLIT_SEED,
            "model": MODEL_ID,
            "train_samples_per_language": TRAIN_SAMPLES_PER_LANGUAGE,
            "test_samples_per_language": TEST_SAMPLES_PER_LANGUAGE,
            "test_split": "test",
            "prompt_protocol": RUN_CONFIG["prompt_protocol"],
        },
        wandb_project=None,
        wandb_mode="disabled",
    )
    matrix = result["ledger"].performance_matrix()
    if len(matrix) != len(LANGUAGES) or any(len(row) != len(LANGUAGES) for row in matrix):
        raise ValueError(f"Seed {seed} produced an incomplete performance matrix.")
    evaluation_json = Path(result["ledger"].json_path)
    evaluation_csv = evaluation_json.with_suffix(".csv")
    if not evaluation_json.is_file() or not evaluation_csv.is_file():
        raise FileNotFoundError(f"Seed {seed} did not produce both evaluation records.")
    seed_results.append({
        "seed": seed,
        "experiment_id": experiment_id,
        "matrix": matrix,
        "metrics": result["metrics"],
        "evaluation_json": str(evaluation_json.relative_to(GROUP_OUTPUT)),
        "evaluation_csv": str(evaluation_csv.relative_to(GROUP_OUTPUT)),
    })
    progress = {
        "experiment_group_id": GROUP_ID,
        "completed_seeds": [item["seed"] for item in seed_results],
        "expected_seeds": SEEDS,
    }
    (GROUP_OUTPUT / "seed_progress.json").write_text(
        json.dumps(progress, indent=2, allow_nan=False) + "\n", encoding="utf-8"
    )
    print(f"Completed seed {seed}. Evaluation log: {evaluation_json}")
    del result, trainer
    gc.collect()
    torch.cuda.empty_cache()

if [item["seed"] for item in seed_results] != SEEDS:
    raise ValueError("Expected validated results for seeds 42, 43, and 44.")
print("All three seed runs completed.")


In [ ]:
import json
import statistics

matrices = [item["matrix"] for item in seed_results]
mean_matrix = [
    [statistics.mean(matrix[row][column] for matrix in matrices) for column in range(len(LANGUAGES))]
    for row in range(len(LANGUAGES))
]
std_matrix = [
    [statistics.stdev(matrix[row][column] for matrix in matrices) for column in range(len(LANGUAGES))]
    for row in range(len(LANGUAGES))
]
seed_pattern_results = []
for item in seed_results:
    matrix = item["matrix"]
    final_row = matrix[-1]
    diagonal = [matrix[index][index] for index in range(len(LANGUAGES))]
    previous_final_mean = statistics.mean(final_row[:-1])
    previous_learned_mean = statistics.mean(diagonal[:-1])
    last_language_strong = final_row[-1] >= previous_final_mean
    earlier_languages_degraded = previous_final_mean < previous_learned_mean
    seed_pattern_results.append({
        "seed": item["seed"],
        "last_seen_language_strong_relative_to_previous_mean": last_language_strong,
        "earlier_languages_degraded_on_average": earlier_languages_degraded,
        "checkpoint_pattern_passed": last_language_strong and earlier_languages_degraded,
    })

final_mean = mean_matrix[-1]
first_seen_mean = [
    statistics.mean(matrix[index][index] for matrix in matrices)
    for index in range(len(LANGUAGES))
]
report = {
    **RUN_CONFIG,
    "mean_performance_matrix_rows_after_each_stage": mean_matrix,
    "std_performance_matrix_rows_after_each_stage": std_matrix,
    "mean_final_accuracies_by_language": dict(zip(LANGUAGES, final_mean)),
    "mean_first_seen_accuracies_by_language": dict(zip(LANGUAGES, first_seen_mean)),
    "per_seed_results": seed_results,
    "per_seed_checkpoint_checks": seed_pattern_results,
    "checkpoint_pattern_pass_rate": sum(
        item["checkpoint_pattern_passed"] for item in seed_pattern_results
    ) / len(seed_pattern_results),
    "checkpoint_pattern_passed_all_seeds": all(
        item["checkpoint_pattern_passed"] for item in seed_pattern_results
    ),
    "interpretation_note": "Three completed seeds provide a preliminary repeatability check, not a definitive scientific conclusion.",
}
summary_path = GROUP_OUTPUT / "multiseed_summary.json"
summary_path.write_text(
    json.dumps(report, indent=2, allow_nan=False) + "\n", encoding="utf-8"
)
print("Summary:", summary_path)
print("Per-seed checkpoint checks:", json.dumps(seed_pattern_results, indent=2))
print("Checkpoint pass rate:", report["checkpoint_pattern_pass_rate"])
if report["checkpoint_pattern_passed_all_seeds"]:
    print("All seeds met the pilot criterion. This is preliminary evidence, not definitive reproduction.")
else:
    print("The three-seed run is complete, but the checkpoint pattern was not met by every seed.")


In [ ]:
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile
from IPython.display import FileLink, display

archive_path = Path("/kaggle/working") / f"{GROUP_ID}_outputs.zip"
with ZipFile(archive_path, "w", compression=ZIP_DEFLATED, compresslevel=6) as archive:
    for path in sorted(GROUP_OUTPUT.rglob("*")):
        if path.is_file():
            archive.write(path, arcname=str(Path(GROUP_ID) / path.relative_to(GROUP_OUTPUT)))
print("Output archive:", archive_path)
print("Size (MiB):", round(archive_path.stat().st_size / (1024 * 1024), 1))
display(FileLink(str(archive_path)))
